# Module 5 - Lab: Generate, Check and Store Data with phyphox

This 45-minute lab follows the early data pipeline from the Module 5 impulse: **generate, check, organise, store**.

The goal is to turn a fresh phyphox export into data that another person can find, understand, and reuse later.

## Timebox
- 10 min: Generate a measurement and export it.
- 10 min: Check whether the recording worked correctly.
- 15 min: Organise files, rename them, and separate raw from preprocessed data.
- 10 min: Store the artefact in hot storage and document the location.

## Module 5 ideas used in this lab
- A raw export is not yet research data.
- Check directly after generation, before analysis begins.
- The check asks whether the run recorded correctly, not whether the result is nice.
- Keep or delete failed runs deliberately.
- Raw data is sacred: never edit it in place.
- Use clear folders, naming conventions, and versions.
- Store hot data in Sciebo and document the path.

## Section 1: Import Libraries

Run this cell first. The notebook uses Python to inspect file formats, check exported data, and create a small storage checklist.

In [ ]:
# Section 1: Import Libraries
from pathlib import Path
import csv
import json
import re
import shutil

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.precision', 4)

print('Libraries imported successfully.')

## Section 2: Generate the Measurement

Use phyphox on your phone to record one run on the model.

Before exporting, write down the context. This is the information that makes the file understandable later.

- What subsystem did you measure: `drivetrain` or `suspension`?
- Which quantity did you record: acceleration, light, rotation, etc.?
- Which phone/sensor did you use?
- What was the intended duration?
- Which run number is this?
- Did anything unusual happen during the run?

Export the phyphox data as CSV or Excel and place the raw export somewhere under `data/`.

## Section 3: Point to the Recorded Data

The shared `metadata.json` file should only contain the path to the recorded data file. Collection metadata stays with the data export itself:

- For Excel/XLS/XLSX, metadata is extracted from the workbook.
- For CSV, this notebook checks whether a `meta/` folder exists next to the CSV file.

Change `recorded_data_path_override` if you exported a new file.

In [ ]:
# Section 3: Point to the Recorded Data
project_root = Path.cwd()
metadata_path = project_root / 'metadata.json'

if metadata_path.exists():
    with metadata_path.open('r', encoding='utf-8') as f:
        pointer_metadata = json.load(f)
else:
    pointer_metadata = {'recorded_data_path': 'data/drivetrain/Example/Raw Data.csv'}

# Example override. Keep None if metadata.json already points to your file.
recorded_data_path_override = None
# recorded_data_path_override = 'data/drivetrain/Example/Raw Data.csv'
# recorded_data_path_override = 'data/suspension/Example/Beschleunigung ohne g.xls'

if recorded_data_path_override:
    pointer_metadata['recorded_data_path'] = recorded_data_path_override

recorded_data_path = pointer_metadata['recorded_data_path']
selected_data_path = project_root / recorded_data_path

save_pointer_metadata = False
if save_pointer_metadata:
    with metadata_path.open('w', encoding='utf-8') as f:
        json.dump({'recorded_data_path': recorded_data_path}, f, indent=2)
    print('Saved pointer metadata:', metadata_path)

print('Recorded data path:', selected_data_path)
print('File exists:', selected_data_path.exists())

## Section 4: Detect File Format and Load Data

One common data-quality problem is inconsistent format: comma vs. semicolon, decimal point vs. decimal comma. This section detects the format before loading.

Supported formats:
- Excel
- CSV comma, decimal point
- CSV tabulator, decimal point
- CSV semicolon, decimal point
- CSV tabulator, decimal comma
- CSV semicolon, decimal comma

In [ ]:
# Section 4: Detect File Format and Load Data
def read_text_sample(path, max_bytes=65536):
    raw = path.read_bytes()[:max_bytes]
    for encoding in ['utf-8-sig', 'utf-8', 'latin1']:
        try:
            return raw.decode(encoding), encoding
        except UnicodeDecodeError:
            pass
    return raw.decode('latin1', errors='replace'), 'latin1'


def detect_csv_format(path):
    sample, encoding = read_text_sample(path)
    lines = [line for line in sample.splitlines() if line.strip()][:30]
    delimiters = [',', '\t', ';']
    delimiter_names = {',': 'comma', '\t': 'tabulator', ';': 'semicolon'}

    best = None
    for delimiter in delimiters:
        parsed = list(csv.reader(lines, delimiter=delimiter))
        widths = [len(row) for row in parsed if row]
        multi_column_rows = [width for width in widths if width > 1]
        common_width = max(set(multi_column_rows), key=multi_column_rows.count) if multi_column_rows else 1
        score = multi_column_rows.count(common_width) * common_width if multi_column_rows else 0
        candidate = (score, common_width, delimiter)
        if best is None or candidate > best:
            best = candidate

    delimiter = best[2]
    parsed = list(csv.reader(lines, delimiter=delimiter))
    tokens = []
    for row in parsed[1:]:
        tokens.extend([item.strip() for item in row])

    decimal_comma = sum(1 for item in tokens if re.fullmatch(r'[-+]?\d+,\d+(?:[eE][-+]?\d+)?', item))
    decimal_point = sum(1 for item in tokens if re.fullmatch(r'[-+]?\d+\.\d+(?:[eE][-+]?\d+)?', item))
    decimal = ',' if decimal_comma > decimal_point else '.'
    decimal_name = 'decimal comma' if decimal == ',' else 'decimal point'

    return {
        'container_format': 'csv',
        'format_label': f"csv ({delimiter_names[delimiter]}, {decimal_name})",
        'delimiter': delimiter,
        'decimal': decimal,
        'encoding': encoding
    }


def read_recorded_data(path):
    suffix = path.suffix.lower()
    if suffix == '.csv':
        detected_format = detect_csv_format(path)
        table = pd.read_csv(
            path,
            sep=detected_format['delimiter'],
            decimal=detected_format['decimal'],
            encoding=detected_format['encoding']
        )
        return table, detected_format
    if suffix in ['.xlsx', '.xls']:
        table = pd.read_excel(path)
        return table, {'container_format': 'excel', 'format_label': 'excel'}
    raise ValueError(f'Unsupported file type: {suffix}')


df_raw, detected_format = read_recorded_data(selected_data_path)

print('Detected format:', detected_format['format_label'])
print('Rows, columns:', df_raw.shape)
display(df_raw.head())

## Section 5: Extract Metadata Stored with the Data

This is the separation used in this course:

- `metadata.json`: only points to the recorded data file.
- The recorded data export or its neighbouring `meta/` folder contains the collection metadata.

For CSV, this cell checks the adjacent `meta/` folder. For Excel, it lists workbook sheets and previews their content.

In [ ]:
# Section 5: Extract Metadata Stored with the Data
def extract_csv_meta_folder(path):
    meta_dir = path.parent / 'meta'
    metadata = {
        'source': 'csv_meta_folder',
        'meta_folder': str(meta_dir.relative_to(project_root)).replace('\\', '/'),
        'exists': meta_dir.exists(),
        'files': {}
    }
    if not meta_dir.exists():
        return metadata

    for file in sorted(meta_dir.glob('*')):
        if not file.is_file() or file.suffix.lower() not in ['.csv', '.xlsx', '.xls']:
            continue
        try:
            frame, fmt = read_recorded_data(file)
            metadata['files'][str(file.relative_to(project_root)).replace('\\', '/')] = {
                'format': fmt,
                'columns': frame.columns.tolist(),
                'row_count': int(len(frame)),
                'preview': frame.head(20).where(pd.notna(frame), None).to_dict(orient='records')
            }
        except Exception as error:
            metadata['files'][str(file.relative_to(project_root)).replace('\\', '/')] = {'error': str(error)}
    return metadata


def extract_excel_metadata(path):
    excel = pd.ExcelFile(path)
    metadata = {'source': 'excel_workbook', 'sheets': excel.sheet_names, 'sheet_previews': {}}
    for sheet in excel.sheet_names:
        preview = pd.read_excel(path, sheet_name=sheet, nrows=20)
        metadata['sheet_previews'][sheet] = {
            'columns': preview.columns.tolist(),
            'row_count_preview': int(len(preview)),
            'preview': preview.head(10).where(pd.notna(preview), None).to_dict(orient='records')
        }
    return metadata


if selected_data_path.suffix.lower() == '.csv':
    recorded_metadata = extract_csv_meta_folder(selected_data_path)
else:
    recorded_metadata = extract_excel_metadata(selected_data_path)

display(pd.json_normalize(recorded_metadata, sep='.').T.rename(columns={0: 'value'}))

## Section 6: Check Correct Recording

This is a **recording check**, not an analysis result.

Ask the Module 5 questions:

- Did the file actually write?
- Is the duration plausible?
- Are the expected columns and axes present?
- Are there missing values, flat lines, clipping, or dropouts?
- Did the generation run as set up?

In [ ]:
# Section 6: Check Correct Recording
numeric_columns = df_raw.select_dtypes(include=[np.number]).columns.tolist()
time_candidates = [c for c in numeric_columns if 'time' in c.lower() or 'zeit' in c.lower()]
time_column = time_candidates[0] if time_candidates else None

quality_checks = []
quality_checks.append({'check': 'file_exists', 'result': selected_data_path.exists(), 'note': str(selected_data_path)})
quality_checks.append({'check': 'has_rows', 'result': len(df_raw) > 0, 'note': f'{len(df_raw)} rows'})
quality_checks.append({'check': 'has_columns', 'result': len(df_raw.columns) > 0, 'note': f'{len(df_raw.columns)} columns'})
quality_checks.append({'check': 'missing_values', 'result': not df_raw.isna().any().any(), 'note': str(df_raw.isna().sum().to_dict())})
quality_checks.append({'check': 'duplicate_rows', 'result': df_raw.duplicated().sum() == 0, 'note': f"{int(df_raw.duplicated().sum())} duplicate rows"})

for col in numeric_columns:
    unique_count = df_raw[col].nunique(dropna=True)
    quality_checks.append({
        'check': f'not_flat_line:{col}',
        'result': unique_count > 1,
        'note': f'{unique_count} unique values'
    })

if time_column:
    time_diff = df_raw[time_column].diff().dropna()
    duration = df_raw[time_column].max() - df_raw[time_column].min()
    quality_checks.append({'check': 'time_increases', 'result': bool((time_diff > 0).all()), 'note': f'time column: {time_column}'})
    quality_checks.append({'check': 'duration_available', 'result': bool(duration > 0), 'note': f'duration: {duration}'})

quality_report = pd.DataFrame(quality_checks)
display(quality_report)

display(df_raw.describe(include='all').T)

## Section 7: Keep or Delete Decision

Decide immediately after the check.

- Keep the run if it recorded correctly.
- Delete or repeat the run if it carries no useful information, for example wrong sensor, wrong axis, missing file, or malfunction.
- Exception: keep a failed run only if it documents a new challenge that should be discussed.

**Decision:** keep / repeat / delete

**Reason:** 

## Section 8: Organise Raw, Preprocessed, Analysed, Docs

Use a shallow structure that another person can navigate.

Recommended structure:

```text
data/<subsystem>/<run_id>/
  raw/
  preprocessed/
  analysed/
  docs/
```

Raw data is sacred. Never edit it in place.

In [ ]:
# Section 8: Organise Raw, Preprocessed, Analysed, Docs
subsystem = 'drivetrain'  # drivetrain or suspension
quantity = 'measurement'  # example: accel, light, rotation
run_number = 1
date_iso = '2026-07-01'
project = 'rdm'
stage = 'raw'
version = 'v0.1.0'

run_id = f'{date_iso}_{project}_{subsystem}_{quantity}_run{run_number:02d}'
target_root = project_root / 'data' / subsystem / run_id
raw_folder = target_root / 'raw'
preprocessed_folder = target_root / 'preprocessed'
analysed_folder = target_root / 'analysed'
docs_folder = target_root / 'docs'

for folder in [raw_folder, preprocessed_folder, analysed_folder, docs_folder]:
    folder.mkdir(parents=True, exist_ok=True)

recommended_filename = f'{run_id}_{stage}_{version}{selected_data_path.suffix.lower()}'
target_raw_path = raw_folder / recommended_filename

print('Run ID:', run_id)
print('Recommended raw filename:', recommended_filename)
print('Target raw path:', target_raw_path)

## Section 9: Copy Raw Export Without Editing It

This creates a copy of the raw export with a clear name. The original export is not modified.

Only set `copy_raw_file = True` after checking the target path.

In [ ]:
# Section 9: Copy Raw Export Without Editing It
copy_raw_file = False

if copy_raw_file:
    if target_raw_path.exists():
        raise FileExistsError(f'Target file already exists. Choose a new version: {target_raw_path}')
    shutil.copy2(selected_data_path, target_raw_path)
    print('Copied raw export to:', target_raw_path)
else:
    print('Dry run only. Set copy_raw_file = True to copy the raw export.')

## Section 10: Create a Documentation File

A README explains the rules in plain words. This one records what was checked, where the data came from, and what storage path should be documented in the DMP.

In [ ]:
# Section 10: Create a Documentation File
sciebo_path = 'sciebo://CHANGE_ME/project/path/to/hot-storage'

lab5_summary = {
    'recorded_data_path': str(selected_data_path.relative_to(project_root)).replace('\\', '/'),
    'detected_format': detected_format,
    'metadata_source': recorded_metadata.get('source'),
    'recommended_run_id': run_id,
    'recommended_raw_path': str(target_raw_path.relative_to(project_root)).replace('\\', '/'),
    'row_count': int(df_raw.shape[0]),
    'column_count': int(df_raw.shape[1]),
    'columns': df_raw.columns.tolist(),
    'quality_checks': quality_report.to_dict(orient='records'),
    'hot_storage_path': sciebo_path,
    'backup_note': 'Hot data should be synced, backed up, shared with the team, and documented in the DMP.'
}

readme_text = f'''# {run_id}

## What was measured
- Subsystem: {subsystem}
- Quantity: {quantity}
- Run number: {run_number}
- Source file: {lab5_summary['recorded_data_path']}
- Detected format: {detected_format['format_label']}

## Data handling
- Raw data is stored separately and must not be edited in place.
- Preprocessed and analysed files are new artefacts and need their own names and versions.
- File names use ISO date, project, subsystem, quantity, run number, stage, and version.

## Hot storage
- Sciebo path: {sciebo_path}
- Document this path in the DMP.

## Recording check
{quality_report.to_string(index=False)}
'''

summary_path = docs_folder / f'{run_id}_lab5_summary.json'
readme_path = docs_folder / 'README.md'

write_docs = False
if write_docs:
    with summary_path.open('w', encoding='utf-8') as f:
        json.dump(lab5_summary, f, indent=2)
    readme_path.write_text(readme_text, encoding='utf-8')
    print('Wrote summary:', summary_path)
    print('Wrote README:', readme_path)
else:
    print('Dry run only. Set write_docs = True to write documentation files.')
    print(readme_text[:1200])

## Section 11: Hot Storage Checklist

Use Sciebo as hot storage for active working data.

- Synced and backed up: a laptop failure must not cost a run.
- Shared with the team: no email attachments as the main data store.
- Path documented: the location goes into the DMP.
- 3-2-1 thinking: three copies, two storage types, one elsewhere. In this course, Sciebo covers the backup side for hot data.

**Sciebo path documented in DMP:** yes / no

**Who can access the folder:**

**Remaining action before analysis:**

## Section 12: Final Lab Check

Before leaving Lab 5, make sure you can answer these questions:

- Did the run record correctly?
- Did you keep, repeat, or delete it for a documented reason?
- Is raw data separate from preprocessed or analysed data?
- Does the filename follow the agreed naming convention?
- Is the version explicit, for example `v0.1.0`?
- Is the Sciebo hot-storage path documented?
- Can another group find this file tomorrow and understand what it is?

The stored artefact is now ready to become input for the following analysis modules.